# Discriminator MI Estimator

This notebook estimates `I(endpoint; trajectory)` by training a discriminator to separate aligned endpoint/trajectory pairs from product-of-marginals pairs. BCE trains the density-ratio classifier; the reported MI curve uses only the Donsker-Varadhan bound computed from held-out positive and shuffled-negative pairs.

## 1. Setup

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import torch
from IPython.display import clear_output, display
from torch import nn
from torch.nn import functional as F

NOTEBOOK_ROOT = Path.cwd()
if (NOTEBOOK_ROOT / "shared").exists():
    sys.path.insert(0, str(NOTEBOOK_ROOT))
elif (NOTEBOOK_ROOT / "notebooks" / "shared").exists():
    sys.path.insert(0, str(NOTEBOOK_ROOT / "notebooks"))

from shared.magentic_field import (
    B,
    DEVICE,
    DTYPE,
    EARTH_RADIUS_M,
    STEP_METERS,
    STEP_RAD,
    exp_map_sphere,
    fibonacci_sphere,
    normalize,
    sample_uniform_sphere,
    tangent_basis,
)
from shared.trajectory_sampler import (
    EPS_B_STD,
    EPS_U_STD,
    FEATURE_DIM,
    FEATURE_NORM_SAMPLES,
    MAX_STEPS,
    make_trajectories,
    estimate_feature_normalization,
)

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"device={DEVICE}, step={STEP_METERS:.1f} m, angular step={STEP_RAD:.3e} rad")

FEATURE_MEAN, FEATURE_STD = estimate_feature_normalization(device=DEVICE)

D_MODEL = 256
N_HEADS = 4
N_LAYERS = 3
DROPOUT = 0.05
TRAIN_EPOCHS = 90
TRAIN_BATCHES_PER_EPOCH = 4
TRAIN_BATCH_SIZE = 512
LEARNING_RATE = 2e-4
NWJ_LEARNING_RATE = 5e-5
WEIGHT_DECAY = 1e-4
PLOT_EVERY = 2
BCE_WARMUP_EPOCHS = 6
NWJ_BCE_WEIGHT = 0.05
NWJ_EXP_CLAMP = 8.0
LOGIT_ABS_CLAMP = 22.0
NEGATIVES_PER_POSITIVE = 8
EVAL_NEGATIVES_PER_POSITIVE = 32
EVAL_LOGIT_CHUNK_SIZE = 1024
NWJ_EARLY_STOP_MIN_EPOCHS = 24
NWJ_EARLY_STOP_PATIENCE = 10
NWJ_EARLY_STOP_MIN_DELTA_BITS = 0.05

RESIDUAL_CLIP_SIGMA = 30.0
ENDPOINT_OFFSET_SCALE_M = 2_000_000.0
PAIR_FEATURE_DIM = FEATURE_DIM + 3 + 1 + 3 + 2
EVAL_STEP_COUNTS = [1, 2, 4, 8, 12, 16]
EVAL_BATCH_SIZE = 1024
EVAL_BATCHES_PER_POINT = 8

## 2. Discriminator

In [ ]:
def log_map_sphere_2d_meters(base, target):
    dot = (target * base).sum(dim=-1).clamp(-1.0, 1.0)
    tangent = target - dot[:, None] * base
    sin_theta = tangent.norm(dim=-1).clamp_min(1e-12)
    theta = torch.atan2(sin_theta, dot)
    direction = tangent / sin_theta[:, None]
    e1, e2 = tangent_basis(base)
    v_m = EARTH_RADIUS_M * theta[:, None] * direction
    return torch.stack([(v_m * e1).sum(dim=-1), (v_m * e2).sum(dim=-1)], dim=-1)


@torch.no_grad()
def reconstruct_candidate_path(endpoint, x, pad_mask):
    batch_size, token_count, _ = x.shape
    positions_rev = []
    u = endpoint
    positions_rev.append(u)
    for t in range(token_count - 2, -1, -1):
        du = x[:, t, 6:8]
        e1, e2 = tangent_basis(u)
        back_step = -du[:, 0:1] * e1 - du[:, 1:2] * e2
        u = exp_map_sphere(u, back_step)
        positions_rev.append(u)
    return torch.stack(list(reversed(positions_rev)), dim=1)


def make_pair_features(x, pad_mask, endpoint):
    candidate_path = reconstruct_candidate_path(endpoint, x, pad_mask)
    predicted_B = B(candidate_path.reshape(-1, 3)).reshape(candidate_path.shape)
    residual_sigma = (x[:, :, :3] - predicted_B) / EPS_B_STD
    residual_B = torch.clamp(residual_sigma / RESIDUAL_CLIP_SIGMA, -1.0, 1.0)
    residual_energy = torch.log1p(residual_sigma.square().sum(dim=-1, keepdim=True))

    last_idx = (~pad_mask).sum(dim=1).sub(1).clamp_min(0)
    final_B = x[torch.arange(x.shape[0], device=x.device), last_idx, :3]
    anchor = normalize(final_B)
    endpoint_offset = (
        log_map_sphere_2d_meters(anchor, endpoint) / ENDPOINT_OFFSET_SCALE_M
    )
    endpoint_B = B(endpoint)

    endpoint_offset_tokens = endpoint_offset[:, None, :].expand(-1, x.shape[1], -1)
    endpoint_B_tokens = endpoint_B[:, None, :].expand(-1, x.shape[1], -1)
    scaled_x = (x - FEATURE_MEAN) / FEATURE_STD
    return torch.cat(
        [
            scaled_x,
            residual_B,
            residual_energy,
            endpoint_B_tokens,
            endpoint_offset_tokens,
        ],
        dim=-1,
    )


class EndpointTrajectoryDiscriminator(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS):
        super().__init__()
        self.input_projection = nn.Sequential(
            nn.Linear(PAIR_FEATURE_DIM, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
        )
        self.position_embedding = nn.Parameter(torch.zeros(1, MAX_STEPS + 1, d_model))
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=DROPOUT,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.attention = nn.Linear(d_model, 1)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 2 * d_model),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(2 * d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, 1),
        )

    def forward(self, x, pad_mask, endpoint):
        pair_features = make_pair_features(x, pad_mask, endpoint)
        h = self.input_projection(pair_features)
        h = h + self.position_embedding[:, : h.shape[1], :]
        h = self.encoder(h, src_key_padding_mask=pad_mask)
        attention_logits = (
            self.attention(h).squeeze(-1).masked_fill(pad_mask, -torch.inf)
        )
        weights = torch.softmax(attention_logits, dim=-1)
        pooled = torch.sum(h * weights[:, :, None], dim=1)
        return self.head(pooled).squeeze(-1)


def sample_product_endpoints(batch_size, negative_count, device=DEVICE):
    return sample_uniform_sphere(batch_size * negative_count, device=device).reshape(
        batch_size, negative_count, 3
    )


def contrastive_logits(
    model,
    x,
    pad_mask,
    endpoint,
    negative_count=NEGATIVES_PER_POSITIVE,
    negative_chunk_size=None,
):
    positive_logit = model(x, pad_mask, endpoint)
    negative_endpoint = sample_product_endpoints(
        endpoint.shape[0], negative_count, endpoint.device
    )
    flat_negative_endpoint = negative_endpoint.reshape(-1, 3)
    if negative_chunk_size is None:
        repeated_x = x.repeat_interleave(negative_count, dim=0)
        repeated_mask = pad_mask.repeat_interleave(negative_count, dim=0)
        negative_logit = model(repeated_x, repeated_mask, flat_negative_endpoint)
    else:
        source_index = torch.arange(x.shape[0], device=x.device).repeat_interleave(
            negative_count
        )
        chunks = []
        for start in range(0, flat_negative_endpoint.shape[0], negative_chunk_size):
            stop = min(start + negative_chunk_size, flat_negative_endpoint.shape[0])
            row_index = source_index[start:stop]
            chunks.append(
                model(
                    x.index_select(0, row_index),
                    pad_mask.index_select(0, row_index),
                    flat_negative_endpoint[start:stop],
                )
            )
        negative_logit = torch.cat(chunks, dim=0)
    return positive_logit, negative_logit


def make_discriminator_batch(batch_size, step_count):
    x, pad_mask, endpoint, _ = make_trajectories(batch_size, step_counts=step_count)
    positive_logit_shape = (batch_size,)
    labels = torch.cat(
        [
            torch.ones(positive_logit_shape, device=DEVICE),
            torch.zeros(batch_size * NEGATIVES_PER_POSITIVE, device=DEVICE),
        ],
        dim=0,
    )
    return x, pad_mask, endpoint, labels


def bounded_logits(logits):
    return logits.clamp(-LOGIT_ABS_CLAMP, LOGIT_ABS_CLAMP)


def discriminator_dv_nats(positive_logits, negative_logits):
    positive_logits = bounded_logits(positive_logits)
    negative_logits = bounded_logits(negative_logits)
    return (
        positive_logits.mean()
        - torch.logsumexp(negative_logits, dim=0)
        + math.log(negative_logits.numel())
    )


def discriminator_nwj_nats(positive_logits, negative_logits):
    positive_logits = bounded_logits(positive_logits)
    negative_exp_arg = (bounded_logits(negative_logits) - 1.0).clamp(max=NWJ_EXP_CLAMP)
    return positive_logits.mean() - torch.exp(negative_exp_arg).mean()


def discriminator_nwj_loss(positive_logits, negative_logits):
    return -discriminator_nwj_nats(positive_logits, negative_logits)


@torch.no_grad()
def discriminator_dv_bits(positive_logits, negative_logits):
    return float(
        (discriminator_dv_nats(positive_logits, negative_logits) / math.log(2.0)).cpu()
    )


@torch.no_grad()
def discriminator_nwj_bits(positive_logits, negative_logits):
    return float(
        (discriminator_nwj_nats(positive_logits, negative_logits) / math.log(2.0)).cpu()
    )


def create_discriminator_bundle():
    model = EndpointTrajectoryDiscriminator().to(device=DEVICE, dtype=DTYPE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=TRAIN_EPOCHS * TRAIN_BATCHES_PER_EPOCH,
        eta_min=0.1 * LEARNING_RATE,
    )
    return model, optimizer, scheduler


_parameter_count_model = EndpointTrajectoryDiscriminator().to(
    device=DEVICE, dtype=DTYPE
)
sum(p.numel() for p in _parameter_count_model.parameters())

In [ ]:
def train_discriminator_for_steps(step_count):
    model, optimizer, scheduler = create_discriminator_bundle()
    history = []
    best_nwj_bits = -float("inf")
    best_dv_bits = -float("inf")
    best_state_dict = None
    best_epoch = None
    stale_nwj_epochs = 0
    model.train()

    for epoch in range(1, TRAIN_EPOCHS + 1):
        if epoch == BCE_WARMUP_EPOCHS + 1:
            for group in optimizer.param_groups:
                group["lr"] = NWJ_LEARNING_RATE
        losses = []
        skipped_batches = 0
        bces = []
        accuracies = []
        pos_logits = []
        neg_logits = []
        phase = "BCE warmup" if epoch <= BCE_WARMUP_EPOCHS else "NWJ maximize"
        for _ in range(TRAIN_BATCHES_PER_EPOCH):
            x, pad_mask, endpoint, labels = make_discriminator_batch(
                TRAIN_BATCH_SIZE, step_count
            )
            positive_logit, negative_logit = contrastive_logits(
                model, x, pad_mask, endpoint, NEGATIVES_PER_POSITIVE
            )
            logits = torch.cat([positive_logit, negative_logit], dim=0)
            bce = F.binary_cross_entropy_with_logits(logits, labels)
            if epoch <= BCE_WARMUP_EPOCHS:
                loss = bce
            else:
                loss = (
                    discriminator_nwj_loss(positive_logit, negative_logit)
                    + NWJ_BCE_WEIGHT * bce
                )

            if not torch.isfinite(loss):
                skipped_batches += 1
                optimizer.zero_grad(set_to_none=True)
                continue
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            if not torch.isfinite(grad_norm):
                skipped_batches += 1
                optimizer.zero_grad(set_to_none=True)
                continue
            optimizer.step()
            scheduler.step()

            with torch.no_grad():
                prediction = logits > 0.0
                losses.append(float(loss.detach().cpu()))
                bces.append(float(bce.detach().cpu()))
                accuracies.append(
                    float((prediction == labels.bool()).float().mean().cpu())
                )
                pos_logits.append(positive_logit.detach())
                neg_logits.append(negative_logit.detach())

        if not pos_logits:
            print(f"all batches skipped at epoch {epoch}; stopping this model")
            break
        pos_logits = torch.cat(pos_logits)
        neg_logits = torch.cat(neg_logits)
        row = dict(
            epoch=epoch,
            phase=phase,
            loss=float(np.mean(losses)),
            bce=float(np.mean(bces)),
            accuracy=float(np.mean(accuracies)),
            skipped_batches=skipped_batches,
            dv_bits=discriminator_dv_bits(pos_logits, neg_logits),
            nwj_bits=discriminator_nwj_bits(pos_logits, neg_logits),
            pos_logit_bits=float((pos_logits.mean() / math.log(2.0)).cpu()),
            neg_logit_bits=float((neg_logits.mean() / math.log(2.0)).cpu()),
        )
        history.append(row)
        if math.isfinite(row["dv_bits"]) and row["dv_bits"] > best_dv_bits:
            best_dv_bits = row["dv_bits"]
            best_epoch = epoch
            best_state_dict = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }

        if epoch > BCE_WARMUP_EPOCHS:
            if row["nwj_bits"] > best_nwj_bits + NWJ_EARLY_STOP_MIN_DELTA_BITS:
                best_nwj_bits = row["nwj_bits"]
                stale_nwj_epochs = 0
            else:
                stale_nwj_epochs += 1
        should_stop = (
            epoch >= NWJ_EARLY_STOP_MIN_EPOCHS
            and stale_nwj_epochs >= NWJ_EARLY_STOP_PATIENCE
        )

        if (
            epoch == 1
            or epoch % PLOT_EVERY == 0
            or epoch == TRAIN_EPOCHS
            or should_stop
        ):
            clear_output(wait=True)
            fig = go.Figure()
            fig.add_trace(
                go.Scatter(
                    x=[r["epoch"] for r in history],
                    y=[r["bce"] for r in history],
                    mode="lines",
                    name="BCE",
                )
            )
            fig.add_trace(
                go.Scatter(
                    x=[r["epoch"] for r in history],
                    y=[r["dv_bits"] for r in history],
                    mode="lines",
                    name="train DV",
                    yaxis="y2",
                )
            )
            fig.add_trace(
                go.Scatter(
                    x=[r["epoch"] for r in history],
                    y=[r["nwj_bits"] for r in history],
                    mode="lines",
                    name="train NWJ",
                    yaxis="y2",
                    line=dict(dash="dot"),
                )
            )
            fig.add_trace(
                go.Scatter(
                    x=[r["epoch"] for r in history],
                    y=[r["accuracy"] for r in history],
                    mode="lines",
                    name="accuracy",
                    yaxis="y3",
                )
            )
            fig.update_layout(
                title=f"Training endpoint/trajectory discriminator, steps={step_count}, epoch {epoch}/{TRAIN_EPOCHS}, {phase}",
                xaxis_title="epoch",
                yaxis_title="BCE",
                yaxis2=dict(title="DV, bits", overlaying="y", side="right"),
                yaxis3=dict(
                    title="accuracy",
                    overlaying="y",
                    side="right",
                    anchor="free",
                    position=0.97,
                    range=[0.0, 1.0],
                ),
                height=390,
                margin=dict(l=40, r=90, t=50, b=40),
            )
            display(fig)

        if should_stop:
            print(
                f"early stop at epoch {epoch}: train_NWJ={row['nwj_bits']:.4f} bits, "
                f"train_DV={row['dv_bits']:.4f} bits, BCE={row['bce']:.5f}, "
                f"best_DV={best_dv_bits:.4f} bits at epoch {best_epoch}, "
                f"skipped={row['skipped_batches']}"
            )
            break

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)
    return {
        "model": model,
        "history": history,
        "step_count": step_count,
        "best_epoch": best_epoch,
        "best_train_dv_bits": best_dv_bits,
    }


trained_classifiers = {}
for steps in EVAL_STEP_COUNTS:
    trained_classifiers[int(steps)] = train_discriminator_for_steps(int(steps))

for steps, result in trained_classifiers.items():
    last = result["history"][-1]
    print(
        f"steps={steps:2d}: BCE={last['bce']:.4f}, acc={last['accuracy']:.4f}, "
        f"train_NWJ={last['nwj_bits']:.4f} bits, train_DV={last['dv_bits']:.4f} bits, "
        f"best_DV={result['best_train_dv_bits']:.4f} bits at epoch {result['best_epoch']}, "
        f"skipped={last['skipped_batches']}"
    )

## 3. Evaluation

In [7]:
@torch.no_grad()
def estimate_classifier_dv(model, step_count):
    model.eval()
    if str(DEVICE).startswith("cuda"):
        torch.cuda.empty_cache()
    positive_logits = []
    negative_logits = []
    for _ in range(EVAL_BATCHES_PER_POINT):
        x, pad_mask, endpoint, _ = make_trajectories(
            EVAL_BATCH_SIZE, step_counts=step_count
        )
        positive_logit, negative_logit = contrastive_logits(
            model,
            x,
            pad_mask,
            endpoint,
            EVAL_NEGATIVES_PER_POSITIVE,
            negative_chunk_size=EVAL_LOGIT_CHUNK_SIZE,
        )
        positive_logits.append(positive_logit.detach())
        negative_logits.append(negative_logit.detach())
    positive_logits = torch.cat(positive_logits)
    negative_logits = torch.cat(negative_logits)
    return dict(
        dv_bits=discriminator_dv_bits(positive_logits, negative_logits),
        pos_logit_bits=float((positive_logits.mean() / math.log(2.0)).cpu()),
        neg_logit_bits=float((negative_logits.mean() / math.log(2.0)).cpu()),
    )


rows = []
for trained_steps, result in trained_classifiers.items():
    model = result["model"]
    for eval_steps in EVAL_STEP_COUNTS:
        metrics = estimate_classifier_dv(model, int(eval_steps))
        rows.append(
            dict(
                trained_steps=int(trained_steps),
                eval_steps=int(eval_steps),
                distance_m=int(eval_steps) * STEP_METERS,
                matched=int(trained_steps) == int(eval_steps),
                **metrics,
            )
        )

for row in rows:
    print(
        f"trained={row['trained_steps']:2d}, eval={row['eval_steps']:2d}, "
        f"distance={row['distance_m']:6.1f} m, DV={row['dv_bits']:.4f} bits, "
        f"pos_logit={row['pos_logit_bits']:.4f} bits, neg_logit={row['neg_logit_bits']:.4f} bits"
    )

trained= 1, eval= 1, distance= 500.0 m, DV=16.1899 bits, pos_logit=37.0357 bits, neg_logit=-9.7763 bits
trained= 1, eval= 2, distance=1000.0 m, DV=32.2163 bits, pos_logit=25.3478 bits, neg_logit=-9.5831 bits
trained= 1, eval= 4, distance=2000.0 m, DV=27.9534 bits, pos_logit=20.3446 bits, neg_logit=-9.5352 bits
trained= 1, eval= 8, distance=4000.0 m, DV=20.2280 bits, pos_logit=11.7355 bits, neg_logit=-9.5007 bits
trained= 1, eval=12, distance=6000.0 m, DV=17.1617 bits, pos_logit=8.4107 bits, neg_logit=-9.4857 bits
trained= 1, eval=16, distance=8000.0 m, DV=15.5385 bits, pos_logit=6.6776 bits, neg_logit=-9.4756 bits
trained= 2, eval= 1, distance= 500.0 m, DV=12.5661 bits, pos_logit=36.8420 bits, neg_logit=-10.0662 bits
trained= 2, eval= 2, distance=1000.0 m, DV=17.0000 bits, pos_logit=36.8499 bits, neg_logit=-9.9527 bits
trained= 2, eval= 4, distance=2000.0 m, DV=31.1565 bits, pos_logit=36.7937 bits, neg_logit=-9.8736 bits
trained= 2, eval= 8, distance=4000.0 m, DV=41.5550 bits, pos_logi

In [8]:
fig = go.Figure()
for trained_steps in sorted({row["trained_steps"] for row in rows}):
    curve = sorted(
        [row for row in rows if row["trained_steps"] == trained_steps],
        key=lambda row: row["eval_steps"],
    )
    fig.add_trace(
        go.Scatter(
            x=[row["eval_steps"] for row in curve],
            y=[row["dv_bits"] for row in curve],
            mode="lines+markers",
            name=f"trained steps={trained_steps}",
            customdata=[[row["distance_m"], row["matched"]] for row in curve],
            hovertemplate=(
                "trained steps="
                + str(trained_steps)
                + "<br>eval steps=%{x}<br>distance=%{customdata[0]:.0f} m"
                + "<br>matched=%{customdata[1]}<br>DV=%{y:.4f} bits<extra></extra>"
            ),
        )
    )

matched_rows = sorted(
    [row for row in rows if row["matched"]], key=lambda row: row["eval_steps"]
)
fig.add_trace(
    go.Scatter(
        x=[row["eval_steps"] for row in matched_rows],
        y=[row["dv_bits"] for row in matched_rows],
        mode="markers+lines",
        name="matched-step envelope",
        line=dict(color="black", width=4),
        marker=dict(color="black", size=9),
    )
)
fig.update_layout(
    title="Discriminator DV estimate by trained model and evaluation step count",
    xaxis=dict(title="evaluation steps", dtick=1),
    yaxis=dict(title="DV lower bound, bits"),
    height=520,
    margin=dict(l=50, r=55, t=50, b=45),
)
fig.show()